# Unsloth ORPO Smoke Test (Colab)

This notebook is a **smoke test**, not full training. It is designed to confirm that the ORPO path runs end-to-end in Google Colab on a tiny fixed dataset before any longer run is attempted.

## Fixed smoke settings

- Model: `unsloth/Qwen3-0.6B`
- Precision: `fp16`
- Quantization: none (`no QLoRA`)
- `max_seq_length=512`
- LoRA `r=8`
- LoRA `alpha=16`
- LoRA `dropout=0.0`
- ORPO `beta=0.1`
- Learning rate: `2e-5`
- `batch_size=1`
- `max_steps=5`

## Fixed smoke dataset

This notebook must use only:

- `tenacious_bench_v0.1/smoke/dummy_orpo_preferences.jsonl`

It must **not** use:

- `training_data/orpo_preferences.jsonl`


## Pass / fail criteria

A run is considered **PASS** only if all of the following are true:

- Exactly 5 smoke records are loaded.
- Training completes exactly 5 optimizer steps.
- Logged loss values are finite (`not NaN`, `not inf`).
- The adapter output folder contains `adapter_config.json`.
- `adapter_config.json` confirms `"r": 8`.
- A small `smoke_result.json` file is written.

A run is considered **FAIL** if you hit an import error, CUDA OOM, non-finite loss, runtime disconnect, or missing adapter artifacts.


## Expected artifacts

After a passing run, the smoke output folder should contain at least:

- `adapter_config.json`
- `adapter_model.safetensors`
- tokenizer files saved by `save_pretrained(...)`
- `smoke_result.json`

## Troubleshooting

- **OOM**: restart the runtime, confirm you are on a T4 GPU or another roughly 16 GB CUDA runtime, and rerun the notebook with the fixed smoke settings. Do not switch to the full dataset.
- **NaN or inf loss**: stop the run and record the failure. Recheck that the dataset is exactly the 5-record smoke set and that `fp16`, `beta=0.1`, and `max_steps=5` stayed unchanged.
- **Import error**: rerun the install cell, then restart the runtime and rerun from the top.
- **`torch.cuda.is_available()` is false even though `nvidia-smi` works**: this usually means Python is using a CPU-only PyTorch build or the current environment is not a Colab CUDA runtime. Use Colab T4 for the intended smoke run.
- **Missing adapter files**: verify the save cell ran, then inspect the smoke output folder and rerun the save and verification cells.


In [ ]:
# Colab dependency install.
# Run this cell first, then use Runtime -> Restart session if Colab asks for it.

%pip install -q --upgrade pip
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q "trl>=0.9.6" "transformers>=4.51.0" accelerate peft datasets bitsandbytes


In [ ]:
import importlib.metadata as metadata
import platform
import sys

packages = ["unsloth", "trl", "transformers", "torch", "datasets", "peft", "accelerate", "bitsandbytes"]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not-installed"

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": versions,
})


In [ ]:
!nvidia-smi

import torch

cuda_available = torch.cuda.is_available()
torch_build = {
    "torch_version": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": cuda_available,
}
print(torch_build)

assert cuda_available, (
    "CUDA is not available in the active Python runtime. "
    "If you are in Colab, switch to a GPU runtime and rerun from the top. "
    "If you are running locally, this usually means your Python environment has a CPU-only PyTorch build. "
    "Also note that this smoke notebook targets a Colab T4-class 16 GB CUDA runtime; an 8 GB local GPU is not the intended target for the fixed fp16 no-QLoRA settings."
)
device_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
total_vram_gb = props.total_memory / (1024 ** 3)
print({"device": device_name, "total_vram_gb": round(total_vram_gb, 2)})


## Repo access

Pick one path:

1. `drive`: mount Google Drive and point to a repo folder you already copied there.
2. `clone`: clone a repo URL directly into Colab.

Keep the smoke dataset inside the repo so the notebook reads the committed path instead of a manually edited copy.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_MODE = "drive"  # "drive" or "clone"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Sales Eval Bench"
CLONE_URL = "https://github.com/<your-org-or-user>/<your-repo>.git"  # replace only if using clone mode
CLONE_DIR = "/content/Sales Eval Bench"

if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    repo_root = Path(DRIVE_REPO_PATH)
elif REPO_MODE == "clone":
    repo_root = Path(CLONE_DIR)
    if not repo_root.exists():
        subprocess.run(["git", "clone", CLONE_URL, str(repo_root)], check=True)
else:
    raise ValueError(f"Unsupported REPO_MODE: {REPO_MODE}")

assert repo_root.exists(), f"Repo root not found: {repo_root}"
os.chdir(repo_root)
print(f"repo_root={repo_root}")


In [ ]:
import json
from datasets import Dataset

SMOKE_DATASET_PATH = repo_root / "tenacious_bench_v0.1" / "smoke" / "dummy_orpo_preferences.jsonl"
FULL_DATASET_PATH = repo_root / "training_data" / "orpo_preferences.jsonl"

assert SMOKE_DATASET_PATH.exists(), f"Smoke dataset not found: {SMOKE_DATASET_PATH}"
assert FULL_DATASET_PATH.exists(), f"Full dataset path missing unexpectedly: {FULL_DATASET_PATH}"
assert "smoke" in SMOKE_DATASET_PATH.parts, "This notebook must use the smoke dataset path."
assert SMOKE_DATASET_PATH != FULL_DATASET_PATH, "This notebook must not point at the full ORPO dataset."

records = [json.loads(line) for line in SMOKE_DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print({"dataset_path": str(SMOKE_DATASET_PATH), "records_loaded": len(records)})
records[:2]


In [ ]:
assert len(records) == 5, f"Expected exactly 5 smoke records, found {len(records)}"

required_keys = {"prompt", "chosen", "rejected"}
for index, record in enumerate(records, start=1):
    missing = required_keys - set(record)
    assert not missing, f"Record {index} is missing keys: {missing}"
    assert record["chosen"] != record["rejected"], f"Record {index} has identical chosen and rejected strings"

smoke_ds = Dataset.from_list([
    {
        "prompt": record["prompt"],
        "chosen": record["chosen"],
        "rejected": record["rejected"],
    }
    for record in records
])

assert len(smoke_ds) == 5
print(smoke_ds)
print(smoke_ds[0])


## Model and LoRA setup

These cells intentionally keep the smoke configuration fixed:

- `unsloth/Qwen3-0.6B`
- full `fp16`
- `load_in_4bit=False` so this is **not** a QLoRA run
- `max_seq_length=512`
- LoRA `r=8`, `alpha=16`, `dropout=0.0`


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-0.6B"
MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

print({"model_name": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "load_in_4bit": False})


In [ ]:
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print({
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
})


## ORPO smoke run

This training block is intentionally tiny. It should run exactly 5 optimizer steps with batch size 1 over the 5-record smoke set.


In [ ]:
import inspect
from trl import ORPOConfig, ORPOTrainer

OUTPUT_DIR = repo_root / "smoke_outputs" / "orpo_unsloth_qwen3_0_6b_fp16_smoke"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orpo_args = ORPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=5,
    warmup_steps=0,
    fp16=True,
    bf16=False,
    logging_steps=1,
    optim="adamw_8bit",
    max_length=512,
    max_prompt_length=256,
    remove_unused_columns=False,
    report_to=[],
)

trainer_kwargs = {
    "model": model,
    "args": orpo_args,
    "train_dataset": smoke_ds,
}

trainer_signature = inspect.signature(ORPOTrainer.__init__)
if "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer

trainer = ORPOTrainer(**trainer_kwargs)
train_result = trainer.train()

print({
    "global_step": trainer.state.global_step,
    "training_loss": getattr(train_result, "training_loss", None),
    "output_dir": str(OUTPUT_DIR),
})


In [ ]:
import math

loss_values = [entry["loss"] for entry in trainer.state.log_history if isinstance(entry, dict) and "loss" in entry]
assert trainer.state.global_step == 5, f"Expected exactly 5 steps, got {trainer.state.global_step}"
assert len(loss_values) >= 1, "No logged loss values were captured."
assert all(math.isfinite(float(loss)) for loss in loss_values), f"Non-finite loss detected: {loss_values}"

max_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
print({
    "loss_values": loss_values,
    "all_finite": True,
    "max_memory_allocated_gb": round(max_memory_gb, 2),
})


In [ ]:
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved adapter and tokenizer to: {OUTPUT_DIR}")


In [ ]:
import datetime as dt

adapter_config_path = OUTPUT_DIR / "adapter_config.json"
adapter_model_path = OUTPUT_DIR / "adapter_model.safetensors"

assert adapter_config_path.exists(), f"Missing adapter_config.json at {adapter_config_path}"
assert adapter_model_path.exists(), f"Missing adapter_model.safetensors at {adapter_model_path}"

adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
assert adapter_config.get("r") == 8, f"Expected adapter_config r=8, got {adapter_config.get('r')}"

smoke_result = {
    "timestamp_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
    "status": "pass",
    "smoke_dataset_path": str(SMOKE_DATASET_PATH),
    "record_count": len(records),
    "model_name": MODEL_NAME,
    "precision": "fp16",
    "qlora": False,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "orpo_beta": 0.1,
    "learning_rate": 2e-5,
    "batch_size": 1,
    "max_steps": 5,
    "global_step": trainer.state.global_step,
    "loss_values": [float(loss) for loss in loss_values],
    "all_losses_finite": True,
    "adapter_config_path": str(adapter_config_path),
    "adapter_model_path": str(adapter_model_path),
    "max_memory_allocated_gb": round(max_memory_gb, 2),
}

smoke_result_path = OUTPUT_DIR / "smoke_result.json"
smoke_result_path.write_text(json.dumps(smoke_result, indent=2), encoding="utf-8")

print(json.dumps(smoke_result, indent=2))


## After the run

Copy the smoke output folder back into your working tree if you ran this notebook outside Drive. Then record the run result in `docs/progress.md` using the date, runtime, result, and any notes such as peak VRAM or failure mode.
